# IP-5 Help Detection Training Pipeline

Bu notebook Colab uzerinde calismak icin hazirlandi.

Akis:
1. Kaggle dataset indir
2. Metin ve etiket kolonlarini otomatik bul
3. Etiketleri `help / not_help` olarak normalize et
4. BERTurk ile fine-tuning yap
5. Modeli kaydet ve test et


In [ ]:
!pip -q install transformers datasets evaluate accelerate scikit-learn pandas kaggle openpyxl

In [ ]:
import os
from google.colab import files

print("kaggle.json dosyasini yukle")
uploaded = files.upload()

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'wb') as f:
    f.write(uploaded['kaggle.json'])
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('Kaggle auth hazir')

In [ ]:
!mkdir -p /content/data
!kaggle datasets download -d ulkutuncerkucuktas/turkey-earthquake-relief-tweets-dataset -p /content/data --unzip
!find /content/data -maxdepth 3 -type f

In [ ]:
import os
import pandas as pd

candidate_files = []
for root, _, files_ in os.walk('/content/data'):
    for f in files_:
        if f.lower().endswith(('.csv', '.xlsx')):
            candidate_files.append(os.path.join(root, f))

print('Bulunan dosyalar:')
for x in candidate_files:
    print(x)

In [ ]:
DATA_PATH = candidate_files[0]

if DATA_PATH.endswith('.csv'):
    df = pd.read_csv(DATA_PATH)
elif DATA_PATH.endswith('.xlsx'):
    df = pd.read_excel(DATA_PATH)
else:
    raise ValueError('Desteklenmeyen format')

print('Kolonlar:', df.columns.tolist())
print('Satir sayisi:', len(df))
display(df.head())

In [ ]:
def find_text_col(columns):
    candidates = ['text', 'tweet', 'tweet_text', 'content', 'message']
    for c in candidates:
        if c in columns:
            return c
    for c in columns:
        low = c.lower()
        if 'text' in low or 'tweet' in low or 'content' in low:
            return c
    return None

def find_label_col(columns):
    candidates = ['label', 'class', 'target', 'category', 'intent']
    for c in candidates:
        if c in columns:
            return c
    for c in columns:
        low = c.lower()
        if 'label' in low or 'class' in low or 'target' in low or 'intent' in low:
            return c
    return None

text_col = find_text_col(df.columns)
label_col = find_label_col(df.columns)

print('text_col:', text_col)
print('label_col:', label_col)

if label_col is not None:
    print(df[label_col].value_counts(dropna=False))
else:
    print('Label kolonu bulunamadi')

In [ ]:
import numpy as np

work = df[[text_col, label_col]].copy()
work.columns = ['text', 'raw_label']

work['text'] = work['text'].fillna('').astype(str).str.strip()
work = work[work['text'] != ''].copy()

def normalize_label(x):
    x = str(x).strip().lower()

    positive_keywords = [
        'help', 'aid', 'rescue request', 'request_help', 'relief_request',
        'urgent help', 'yardim', 'yardım', 'acil', 'enkaz', 'ses var'
    ]
    negative_keywords = [
        'not_help', 'other', 'irrelevant', 'news', 'prayer', 'support',
        'donation', 'general', 'awareness'
    ]

    if x in {'1', 'help', 'positive', 'request_help', 'urgent_help'}:
        return 1
    if x in {'0', 'not_help', 'negative', 'other', 'irrelevant'}:
        return 0

    if any(k in x for k in positive_keywords):
        return 1
    if any(k in x for k in negative_keywords):
        return 0

    return np.nan

work['label'] = work['raw_label'].apply(normalize_label)
work = work.dropna(subset=['label']).copy()
work['label'] = work['label'].astype(int)

print(work['label'].value_counts(dropna=False))
display(work.head(10))

In [ ]:
import re

def clean_text(text):
    text = str(text)
    text = re.sub(r'http\S+', ' ', text)
    text = re.sub(r'@\S+', ' ', text)
    text = re.sub(r'#', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

work['text'] = work['text'].apply(clean_text)
work = work[work['text'].str.len() > 5].copy()
work = work.drop_duplicates(subset=['text']).reset_index(drop=True)

print('Temizlenmis veri boyutu:', work.shape)
print(work['label'].value_counts().to_dict())
display(work.sample(min(10, len(work)), random_state=42))

In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    work,
    test_size=0.2,
    random_state=42,
    stratify=work['label']
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    stratify=temp_df['label']
)

print('train:', train_df.shape, train_df['label'].value_counts().to_dict())
print('val:', val_df.shape, val_df['label'].value_counts().to_dict())
print('test:', test_df.shape, test_df['label'].value_counts().to_dict())

In [ ]:
from datasets import Dataset, DatasetDict

dataset = DatasetDict({
    'train': Dataset.from_pandas(train_df[['text', 'label']], preserve_index=False),
    'validation': Dataset.from_pandas(val_df[['text', 'label']], preserve_index=False),
    'test': Dataset.from_pandas(test_df[['text', 'label']], preserve_index=False),
})

dataset

In [ ]:
MODEL_NAME = 'dbmdz/bert-base-turkish-cased'
MAX_LEN = 160
SEED = 42
OUTPUT_DIR = '/content/help_berturk_model'

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch['text'],
        truncation=True,
        padding='max_length',
        max_length=MAX_LEN,
    )

tokenized_ds = dataset.map(tokenize, batched=True)
tokenized_ds = tokenized_ds.remove_columns(['text'])
tokenized_ds = tokenized_ds.rename_column('label', 'labels')
tokenized_ds.set_format('torch')

print(tokenized_ds)

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='binary', zero_division=0
    )
    acc = accuracy_score(labels, preds)

    return {
        'accuracy': acc,
        'precision': precision,
        'recall': recall,
        'f1': f1,
    }

In [ ]:
import torch
from torch import nn
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

label_counts = train_df['label'].value_counts().to_dict()
neg_count = label_counts.get(0, 1)
pos_count = label_counts.get(1, 1)

class_weights = torch.tensor(
    [1.0, neg_count / max(pos_count, 1)],
    dtype=torch.float
)

print('class_weights:', class_weights)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get('labels')
        outputs = model(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
        )
        logits = outputs.get('logits')
        loss_fct = nn.CrossEntropyLoss(weight=class_weights.to(logits.device))
        loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    num_train_epochs=4,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    report_to='none',
    seed=SEED,
)

In [ ]:
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds['train'],
    eval_dataset=tokenized_ds['validation'],
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
val_metrics = trainer.evaluate(tokenized_ds['validation'])
test_metrics = trainer.evaluate(tokenized_ds['test'])

print('VAL:', val_metrics)
print('TEST:', test_metrics)

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print('Model kaydedildi:', OUTPUT_DIR)

In [ ]:
from transformers import pipeline

clf = pipeline(
    'text-classification',
    model=OUTPUT_DIR,
    tokenizer=OUTPUT_DIR,
    device=0 if torch.cuda.is_available() else -1
)

samples = [
    "Hatay Antakya'da enkaz altında insanlar var, lütfen yardım edin.",
    'Türkiye için dua ediyoruz.',
    "Adıyaman'da çadır ve su ihtiyacı var.",
    'Bağış yapmak için aşağıdaki linke gidin.'
]

for s in samples:
    print(s)
    print(clf(s))
    print('-' * 50)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!cp -r /content/help_berturk_model /content/drive/MyDrive/help_berturk_model
print('Drivea kopyalandi')